# Setup Silver tables

In [0]:
from utils.logger import get_logger
from utils.ddl import create_silver_schema
from etl_config.constants_config import WATERMARK_TABLE, QUARANTINE_TABLE
from etl_config.silver_config import SILVER_CONFIG

logger = get_logger("silver_setup")

In [0]:
for table_key, cfg in SILVER_CONFIG.items():
    table_name = cfg.target_table
    
    if spark.catalog.tableExists(table_name):
        logger.info(f"Table already exists: {table_name}")
        continue
    
    logger.info(f"Creating silver table: {table_name}")
    
    # Create schema
    schema = create_silver_schema(cfg)
    
    # Create table
    df = spark.createDataFrame([], schema)
    df.write.format("delta").saveAsTable(table_name)
    
    # Set properties
    spark.sql(
        f"""
            ALTER TABLE {table_name}
            SET TBLPROPERTIES (
                'delta.autoOptimize.optimizeWrite' = 'true',
                'delta.autoOptimize.autoCompact' = 'true',
                'description' = :table_description
        )""",
        args={"table_description": cfg.table_description}
    )
    
    # Add column comments
    for column, comment in cfg.column_comments.items():
        try:
            spark.sql(
                f"""
                    COMMENT ON COLUMN {table_name}.{column}
                    IS :comment
                """,
                args={"comment": comment}
            )
            logger.info(f"Added comment to column: {table_name}.{column}")
        except Exception as e:
            logger.warning(f"Could not add comment for {table_name}.{column}: {e}")
    
    logger.info(f"Created: {table_name}")

logger.info("Silver table setup complete.")

### Watermark table

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {WATERMARK_TABLE} (
        table_name   STRING    NOT NULL,
        batch_id     STRING    NOT NULL,
        ingested_at  TIMESTAMP,
        processed_at TIMESTAMP,
        rows_merged  LONG,
        status       STRING    NOT NULL
    )
    USING DELTA
    COMMENT 'Tracking of processed Bronze batches per table'
""")
logger.info(f"Watermark table created/verified: {WATERMARK_TABLE}")

spark.sql(f"""
    ALTER TABLE {WATERMARK_TABLE}
    SET TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact' = 'true'
    )
""")
logger.info(f"Table properties applied for: {WATERMARK_TABLE}")

### Quarantine table

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {QUARANTINE_TABLE} (
        table_name       STRING    NOT NULL,
        batch_id         STRING,
        rejected_at      TIMESTAMP,
        error_type       STRING    NOT NULL,
        _missing_columns ARRAY<STRING>,
        raw_data         STRING
    )
    USING DELTA
    COMMENT 'Bad rows due to missing or uncastable values'
""")
logger.info(f"Quarantine table created/verified: {QUARANTINE_TABLE}")

spark.sql(f"""
    ALTER TABLE {QUARANTINE_TABLE}
    SET TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact' = 'true'
    )
""")
logger.info(f"Table properties applied for: {QUARANTINE_TABLE}")

In [0]:
%sql
use acme_catalog.silver;

select * from information_schema.tables where table_schema = 'silver' order by table_name;

select * from information_schema.columns where table_schema = 'silver' and table_name = 'orders' order by ordinal_position;

